In [ ]:
# =======================================================================
# 📖 데이터셋: SJ-Donald/kor-hate-sentence
# 🌟 데이터셋 의미: 한국어 증오 발언(Hate Speech) 탐지 데이터셋
# 🔎 설명: 이 데이터셋은 인터넷에서 발견되는 증오 발언이나 혐오 표현을 학습하고 분류하는 데 사용됩니다.
#         '문장'은 실제 텍스트, 'hate'나 'labels'는 해당 문장의 혐오 정도(레이블)를 나타내는 정수 값입니다.
#         AI 모델이 어떤 문장이 '나쁜 말'인지 구별하는 능력을 키우는 것이 목표랍니다!
# 💡 실습 목표: 실제 데이터의 일부를 샘플링하여, 혐오 발언의 경향성을 눈으로 직접 확인하고 간단한 키워드 분석을 해봅시다.
# =======================================================================

import random
from datasets import load_dataset
import nltk
from nltk.corpus import stopwords
from collections import Counter

# NLTK 다운로드 (한글 처리를 위한 준비)
try:
    nltk.download('stopwords', language='korean')
except Exception as e:
    print(f"ℹ️ NLTK 다운로드 중 오류 발생: {e}. 인터넷 연결 상태를 확인해주세요.")


# --- [설정 변수] ---
DATASET_NAME = "SJ-Donald/kor-hate-sentence"
TARGET_SPLIT = 'train'
SAMPLE_COUNT = 50  # 분석에 사용할 샘플 개수 (50개만 봐도 충분히 재미있어요!)
# ----------------------

print("💖 ✨ 안녕하세요! 코딩 튜터 AI가 안내하는 데이터 탐험 시간이에요! ✨ 💖")
print(f"🔎 오늘의 데이터셋: {DATASET_NAME} (혐오 발언 분석)")

# 1. 데이터셋 로드 및 안정성 확보 (feat. 에러 처리)
dataset = None
print("\n[STEP 1] 데이터셋 로드 준비...")

try:
    # 🚀 1차 시도: 스트리밍 모드 (가장 빠르고 메모리 효율적!)
    dataset = load_dataset(DATASET_NAME, split=TARGET_SPLIT, streaming=True)
    print("✅ 성공! 스트리밍 모드로 데이터셋을 로드했습니다. 메모리 걱정은 NO!")
except Exception as e:
    print(f"⚠️ 경고: 스트리밍 로드 중 오류가 발생했습니다. ({type(e).__name__}).")
    print("⚡️ 대안으로, 소량의 데이터를 메모리에 로드(Streaming=False)하여 진행할게요.")
    try:
        # 📚 2차 시도: 스트리밍 실패 시, 소량 다운로드 방식으로 전환
        dataset = load_dataset(DATASET_NAME, split=TARGET_SPLIT, streaming=False)
    except Exception as e_fallback:
        print(f"🚨 치명적인 오류: 데이터셋 로드 실패. ({e_fallback}). 스크립트를 종료합니다.")
        exit()


# 2. 샘플링 및 데이터 전처리 (Constraint 적용!)
print("\n[STEP 2] 데이터 샘플링 및 준비...")
if hasattr(dataset, "take"):
    # .take() 메서드를 사용하여 스트리밍 데이터셋(IterableDataset)의 상위 K개만 가져옵니다.
    print(f"✨ 데이터셋이 스트리밍 모드입니다. 상위 {SAMPLE_COUNT}개의 샘플을 준비합니다.")
    sample_data_list = list(dataset.take(SAMPLE_COUNT))
else:
    # 일반 데이터셋 (Dataset)인 경우, list()로 상위 K개만 가져옵니다.
    sample_data_list = list(dataset.select(range(SAMPLE_COUNT)))


print(f"🎉 준비 완료! 총 {len(sample_data_list)}개의 샘플을 분석할 수 있어요.")
print("📊 데이터 구조를 한번 살펴볼까요? 각 샘플은 '문장', 'hate', 'clean', 'labels' 특징을 가지고 있어요.")

# 3. 창의적 실습: 혐오 키워드 정량 분석 및 레이블 분포 탐색
print("\n" + "="*70)
print("🧐 [STEP 3] 초보자를 위한 데이터 인사이트 도출 (정량적 분석)")
print("="*70)

# 3-1. 혐오도(hate) 레이블 분포 분석
# 데이터셋의 정수형 'hate' 값을 통해, 어떤 정도의 혐오 발언이 주로 발견되는지 살펴봅시다.
hate_values = [sample['hate'] for sample in sample_data_list]
hate_counts = Counter(hate_values)

print("\n⭐ 💖 혐오도(Hate) 레이블 분포 분석:")
if hate_counts:
    for hate_level, count in sorted(hate_counts.items(), reverse=True):
        print(f"  - [Hate Level {hate_level}]: 약 {count}번 발견 (이 값이 높을수록 혐오 정도가 높아요!)")
else:
    print("  - 분석할 혐오도 레이블이 충분하지 않습니다.")


# 3-2. 키워드 추출 및 분석 (가장 재미있는 부분!)
print("\n✨ 키워드 탐험: 데이터에 어떤 단어들이 자주 등장할까요?")
all_text = []
for sample in sample_data_list:
    all_text.append(sample['문장'])

# 기본적인 불용어(Stopwords) 제거 및 토큰화
# 실제 프로덕션에서는 더 복잡한 전처리(정규화, 형태소 분석기 사용 등)가 필요하지만, 초보자용 실습이라 간단하게 진행합니다!
stop_words = set(stopwords.words('korean'))
tokenized_corpus = []

for text in all_text:
    # 간단한 공백 분리(토큰화) 후, 불용어 제거
    tokens = [word for word in text.split() if word.strip() and word.strip() not in stop_words]
    tokenized_corpus.append(tokens)

# 전체 코퍼스에서 모든 단어 카운트
all_words = [word for tokens in tokenized_corpus for word in tokens]
word_counts = Counter(all_words)

print("\n📚 빈도수 상위 10개 키워드 (단순 분석):")
most_common_words = word_counts.most_common(10)

# 분석 결과 출력 (가장 재미있는 결과 위주로!)
for word, count in most_common_words:
    print(f"  - '{word}': 약 {count}회")

print("\n🎀 튜터의 Tip!:")
print("위의 키워드들은 이 데이터셋에 자주 등장하는 단어들이에요. 하지만 이 중에서 어떤 단어들이 '혐오'와 연관되는지 패턴을 찾는 것이 AI의 핵심 과제랍니다! 😊")

# 4. 샘플 예시 출력 (결과 확인)
print("\n" + "="*70)
print("🔍 [STEP 4] 분석 샘플 3가지 확인하기:")
print("="*70)
for i in range(min(3, len(sample_data_list))):
    sample = sample_data_list[i]
    print(f"\n[Sample #{i+1}]")
    print(f"  📝 원문: {sample['문장']}")
    print(f"  🏷️ 혐오 레이블 (Int): {sample['hate']} (높을수록 부정적!)")
    print(f"  ✨ 분석 포인트: 이 문장과 레이블을 연결하는 패턴을 찾아보는 것이 딥러닝의 기본 원리예요!")

print("\n🎉👏 축하합니다! 데이터를 로드하고 구조를 분석하는 첫걸음을 성공적으로 마쳤어요!")
print("다음 단계에서는 이 패턴을 이용해 '분류 모델'을 만들어 볼 수 있답니다. 화이팅!")